**Generative AI Use**: For the purposes of the assignments, the use of generative AI is subject to the same policies regarding collaboration. Just as with other collaborators, each student must write down the solutions independently of the output of the interaction and the submission should include a note denoting the nature of the collaboration. The use of generative AI tools to substantially complete sections of the assignments is not in line with the spirit of the assignments, and would be a violation of the [Honor Code](https://communitystandards.stanford.edu/policies-and-guidance/honor-code).

---

**中文对照（参考译文，不含解答）**

**生成式 AI 的使用**：本作业对生成式 AI 的使用采用与协作相同的规则。与其他协作者交流之后，每位学生仍须独立写出自己的解答，不能直接依赖交流中得到的输出；提交时应附上说明，如实记录协作的性质。使用生成式 AI 实质性地完成作业的大部分内容，不符合本作业的教学目的，也会违反[学术诚信准则](https://communitystandards.stanford.edu/policies-and-guidance/honor-code)。

In [ ]:
# CS231n local environment setup (exercise code unchanged)
# [中文] CS231n 本地环境设置 (exercise code unchanged)
from pathlib import Path
import os
import sys

_candidates = [Path.cwd(), *Path.cwd().parents]
_workspace = next((p for p in _candidates if (p / "requirements-local.txt").is_file()
                   and (p / "assignment1" / "cs231n").is_dir()), None)
if _workspace is None:
    raise RuntimeError("Launch Jupyter from the CS231n workspace root.")
ASSIGNMENT_DIR = _workspace / "assignment1"
os.chdir(ASSIGNMENT_DIR)
if str(ASSIGNMENT_DIR) not in sys.path:
    sys.path.insert(0, str(ASSIGNMENT_DIR))
print("Python:", sys.executable)
print("Working directory:", Path.cwd())


# k-Nearest Neighbor (kNN) exercise

*Complete and hand in this completed worksheet (including its outputs and any supporting code outside of the worksheet) with your assignment submission. For more details see the [assignments page](http://vision.stanford.edu/teaching/cs231n/assignments.html) on the course website.*

The kNN classifier consists of two stages:

- During training, the classifier takes the training data and simply remembers it
- During testing, kNN classifies every test image by comparing to all training images and transfering the labels of the k most similar training examples
- The value of k is cross-validated

In this exercise you will implement these steps and understand the basic Image Classification pipeline, cross-validation, and gain proficiency in writing efficient, vectorized code.

---

**中文对照（参考译文，不含解答）**

# k 近邻（kNN）练习

*请完成本 notebook，并将它连同运行输出及 notebook 外的配套代码一起提交。详情见课程网站的[作业页面](http://vision.stanford.edu/teaching/cs231n/assignments.html)。*

kNN 分类器包括两个阶段：

- 训练时：接收训练数据，并把这些数据保存下来。
- 测试时：将每张测试图片与所有训练图片比较，找出最相似的 k 个训练样本，根据它们的标签进行分类。
- k 的取值通过交叉验证确定。

在本练习中，你需要实现这些步骤，理解基本的图像分类流程和交叉验证，并练习编写高效的 NumPy 向量化代码。

In [ ]:
# Run some setup code for this notebook.
# [中文] 运行这个笔记本的设置代码。

import random
import numpy as np
from cs231n.data_utils import load_CIFAR10
import matplotlib.pyplot as plt

# This is a bit of magic to make matplotlib figures appear inline in the notebook
# rather than in a new window.
# [中文] 这有点神奇,让 Mattlotlib 数字在笔记本中出现,而不是在新窗口中出现。
%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # set default size of plots
# [中文] 设置默认的绘图大小
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# Some more magic so that the notebook will reload external python modules;
# see http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
# [中文] 使笔记本能重新装入外部 python 模块的更多魔术;
# 见http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

In [ ]:
# Load the raw CIFAR-10 data.
# [中文] 装入原始的CIFAR - 10 数据。
cifar10_dir = 'cs231n/datasets/cifar-10-batches-py'

# Cleaning up variables to prevent loading data multiple times (which may cause memory issue)
# [中文] 清理变量以防止多次装入数据 (which may cause memory issue)
try:
   del X_train, y_train
   del X_test, y_test
   print('Clear previously loaded data.')
except:
   pass

X_train, y_train, X_test, y_test = load_CIFAR10(cifar10_dir)

# As a sanity check, we print out the size of the training and test data.
# [中文] 作为精神检查,我们打印出 训练和测试数据的大小。
print('Training data shape: ', X_train.shape)
print('Training labels shape: ', y_train.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

In [ ]:
# Visualize some examples from the dataset.
# We show a few examples of training images from each class.
# [中文] 从数据集中可视化一些实例。 我们展示了每个班级的一些训练图像实例。
classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
num_classes = len(classes)
samples_per_class = 7
for y, cls in enumerate(classes):
    idxs = np.flatnonzero(y_train == y)
    idxs = np.random.choice(idxs, samples_per_class, replace=False)
    for i, idx in enumerate(idxs):
        plt_idx = i * num_classes + y + 1
        plt.subplot(samples_per_class, num_classes, plt_idx)
        plt.imshow(X_train[idx].astype('uint8'))
        plt.axis('off')
        if i == 0:
            plt.title(cls)
plt.show()

In [ ]:
# Subsample the data for more efficient code execution in this exercise
# [中文] 在此练习中分样数据, 以便更有效地执行代码
num_training = 5000
mask = list(range(num_training))
X_train = X_train[mask]
y_train = y_train[mask]

num_test = 500
mask = list(range(num_test))
X_test = X_test[mask]
y_test = y_test[mask]

# Reshape the image data into rows
# [中文] 将图像数据重塑为行
X_train = np.reshape(X_train, (X_train.shape[0], -1))
X_test = np.reshape(X_test, (X_test.shape[0], -1))
print(X_train.shape, X_test.shape)

In [ ]:
from cs231n.classifiers import KNearestNeighbor

# Create a kNN classifier instance.
# Remember that training a kNN classifier is a noop:
# the Classifier simply remembers the data and does no further processing
# [中文] 创建 kNN 分类实例。 记住, 训练 kNN分类器是一个切换符: 分类器只记住数据, 不再处理
classifier = KNearestNeighbor()
classifier.train(X_train, y_train)

We would now like to classify the test data with the kNN classifier. Recall that we can break down this process into two steps:

1. First we must compute the distances between all test examples and all train examples.
2. Given these distances, for each test example we find the k nearest examples and have them vote for the label

Lets begin with computing the distance matrix between all training and test examples. For example, if there are **Ntr** training examples and **Nte** test examples, this stage should result in a **Nte x Ntr** matrix where each element (i,j) is the distance between the i-th test and j-th train example.

**Note: For the three distance computations that we require you to implement in this notebook, you may not use the np.linalg.norm() function that numpy provides.**

First, open `cs231n/classifiers/k_nearest_neighbor.py` and implement the function `compute_distances_two_loops` that uses a (very inefficient) double loop over all pairs of (test, train) examples and computes the distance matrix one element at a time.

---

**中文对照（参考译文，不含解答）**

现在使用 kNN 分类器对测试数据进行分类。这个过程可以分为两步：

1. 计算所有测试样本与所有训练样本之间的距离。
2. 根据距离，为每个测试样本找到 k 个最近的训练样本，由它们投票决定标签。

先计算测试样本与训练样本之间的距离矩阵。例如，有 **Ntr** 个训练样本和 **Nte** 个测试样本时，距离矩阵的形状应为 **Nte × Ntr**。其中元素 (i, j) 表示第 i 个测试样本与第 j 个训练样本之间的距离。

**注意：本 notebook 要求实现三种距离计算方式，这三种实现都不允许使用 NumPy 提供的 `np.linalg.norm()`。**

首先打开 `cs231n/classifiers/k_nearest_neighbor.py`，实现 `compute_distances_two_loops`。这个版本对（测试样本、训练样本）对使用双重循环，逐个计算距离矩阵中的元素；它的效率较低。

In [ ]:
# Open cs231n/classifiers/k_nearest_neighbor.py and implement
# compute_distances_two_loops.
# [中文] 打开 cs231n/ classters/ k_nearest_neighbor.py 并安装 compute_distances_two_loops。

# Test your implementation:
# [中文] 测试您的执行:
dists = classifier.compute_distances_two_loops(X_test)
print(dists.shape)

In [ ]:
# We can visualize the distance matrix: each row is a single test example and
# its distances to training examples
# [中文] 我们可以想象出距离矩阵:每行是一个测试示例,它与训练示例的距离
plt.imshow(dists, interpolation='none')
plt.show()

**Inline Question 1**

Notice the structured patterns in the distance matrix, where some rows or columns are visibly brighter. (Note that with the default color scheme black indicates low distances while white indicates high distances.)

- What in the data is the cause behind the distinctly bright rows?
- What causes the columns?

$\color{blue}{\textit Your Answer:}$ *fill this in.*



---

**中文对照（参考译文，不含解答）**

**随堂问题 1**

注意距离矩阵中的结构性图案：某些行或列明显更亮。（默认配色中，黑色表示距离小，白色表示距离大。）

- 数据中的什么因素造成了明显较亮的行？
- 什么因素造成了较亮的列？

**你的回答：** [请自行填写]

In [ ]:
# Now implement the function predict_labels and run the code below:
# We use k = 1 (which is Nearest Neighbor).
# [中文] 现在执行函数 predict_labels 并运行以下代码: 我们使用 k = 1 (which is Nearest Neighbor)。
y_test_pred = classifier.predict_labels(dists, k=1)

# Compute and print the fraction of correctly predicted examples
# [中文] 计算并打印正确预测的示例的分数
num_correct = np.sum(y_test_pred == y_test)
accuracy = float(num_correct) / num_test
print('Got %d / %d correct => accuracy: %f' % (num_correct, num_test, accuracy))

You should expect to see approximately `27%` accuracy. Now lets try out a larger `k`, say `k = 5`:

---

**中文对照（参考译文，不含解答）**

你应当看到约 `27%` 的准确率。现在尝试更大的 k，例如 `k = 5`。

In [ ]:
y_test_pred = classifier.predict_labels(dists, k=5)
num_correct = np.sum(y_test_pred == y_test)
accuracy = float(num_correct) / num_test
print('Got %d / %d correct => accuracy: %f' % (num_correct, num_test, accuracy))

You should expect to see a slightly better performance than with `k = 1`.

---

**中文对照（参考译文，不含解答）**

你应当看到性能比 `k = 1` 略有提升。

**Inline Question 2**

We can also use other distance metrics such as L1 distance.
For pixel values $p_{ij}^{(k)}$ at location $(i,j)$ of some image $I_k$,

the mean $\mu$ across all pixels over all images is $$\mu=\frac{1}{nhw}\sum_{k=1}^n\sum_{i=1}^{h}\sum_{j=1}^{w}p_{ij}^{(k)}$$
And the pixel-wise mean $\mu_{ij}$ across all images is
$$\mu_{ij}=\frac{1}{n}\sum_{k=1}^np_{ij}^{(k)}.$$
The general standard deviation $\sigma$ and pixel-wise standard deviation $\sigma_{ij}$ is defined similarly.

Which of the following preprocessing steps will not change the performance of a Nearest Neighbor classifier that uses L1 distance? Select all that apply. To clarify, both training and test examples are preprocessed in the same way.

1. Subtracting the mean $\mu$ ($\tilde{p}_{ij}^{(k)}=p_{ij}^{(k)}-\mu$.)
2. Subtracting the per pixel mean $\mu_{ij}$  ($\tilde{p}_{ij}^{(k)}=p_{ij}^{(k)}-\mu_{ij}$.)
3. Subtracting the mean $\mu$ and dividing by the standard deviation $\sigma$.
4. Subtracting the pixel-wise mean $\mu_{ij}$ and dividing by the pixel-wise standard deviation $\sigma_{ij}$.
5. Rotating the coordinate axes of the data, which means rotating all the images by the same angle. Empty regions in the image caused by rotation are padded with a same pixel value and no interpolation is performed.

$\color{blue}{\textit Your Answer:}$


$\color{blue}{\textit Your Explanation:}$


---

**中文对照（参考译文，不含解答）**

**随堂问题 2**

还可以使用其他距离度量，例如 L1 距离。对于图像 $I_k$ 中位置 $(i,j)$ 处的像素值 $p_{ij}^{(k)}$，

所有图像的所有像素的总体均值 $\mu$ 为：
$$\mu=\frac{1}{nhw}\sum_{k=1}^n\sum_{i=1}^{h}\sum_{j=1}^{w}p_{ij}^{(k)}$$

所有图像在各个像素位置上的均值 $\mu_{ij}$ 为：
$$\mu_{ij}=\frac{1}{n}\sum_{k=1}^np_{ij}^{(k)}.$$

总体标准差 $\sigma$ 和逐像素位置的标准差 $\sigma_{ij}$ 也以类似方式定义。

以下哪些预处理步骤不会改变采用 L1 距离的最近邻分类器的表现？请选择所有适用选项。训练样本与测试样本采用相同的预处理方式。

1. 减去总体均值 $\mu$（$\tilde{p}_{ij}^{(k)}=p_{ij}^{(k)}-\mu$）。
2. 减去逐像素位置的均值 $\mu_{ij}$（$\tilde{p}_{ij}^{(k)}=p_{ij}^{(k)}-\mu_{ij}$）。
3. 减去总体均值 $\mu$，再除以总体标准差 $\sigma$。
4. 减去逐像素位置的均值 $\mu_{ij}$，再除以逐像素位置的标准差 $\sigma_{ij}$。
5. 旋转数据的坐标轴，也就是把所有图像旋转相同角度。旋转产生的空白区域用相同的像素值填充，并且不进行插值。

**你的回答：** [请自行填写]

**你的解释：** [请自行填写]

In [ ]:
# Now lets speed up distance matrix computation by using partial vectorization
# with one loop. Implement the function compute_distances_one_loop and run the
# code below:
# [中文] 现在让我们使用一个循环的局部矢量来加速距离矩阵计算。 执行函数 compute_distances_one_loop 并运行下面的代码:
dists_one = classifier.compute_distances_one_loop(X_test)

# To ensure that our vectorized implementation is correct, we make sure that it
# agrees with the naive implementation. There are many ways to decide whether
# two matrices are similar; one of the simplest is the Frobenius norm. In case
# you haven't seen it before, the Frobenius norm of two matrices is the square
# root of the squared sum of differences of all elements; in other words, reshape
# the matrices into vectors and compute the Euclidean distance between them.
# [中文] 为了确保我们的矢量化执行正确, 我们确保它与朴素实现一致。 有很多方法可以决定两个矩阵是否相似; 最简单的方法之一是 Frobenius 规范。
# 如果您以前没见过, 两个矩阵的 Frobenius 规范是所有元素差异的平方和的平方根; 换句话说, 将矩阵重塑为矢量, 并计算它们之间的欧立底距离。
difference = np.linalg.norm(dists - dists_one, ord='fro')
print('One loop difference was: %f' % (difference, ))
if difference < 0.001:
    print('Good! The distance matrices are the same')
else:
    print('Uh-oh! The distance matrices are different')

In [ ]:
# Now implement the fully vectorized version inside compute_distances_no_loops
# and run the code
# [中文] 现在在 compute_distances_no_loops 内执行完全矢量化版本并运行代码
dists_two = classifier.compute_distances_no_loops(X_test)

# check that the distance matrix agrees with the one we computed before:
# [中文] 检查距离矩阵是否同意我们之前计算到的:
difference = np.linalg.norm(dists - dists_two, ord='fro')
print('No loop difference was: %f' % (difference, ))
if difference < 0.001:
    print('Good! The distance matrices are the same')
else:
    print('Uh-oh! The distance matrices are different')

In [ ]:
# Let's compare how fast the implementations are
# [中文] 我们来比较一下 执行速度有多快
def time_function(f, *args):
    """
    Call a function f with args and return the time (in seconds) that it took to execute.
    

    中文对照：
    调用带有参数的函数 f, 并返回要执行的时间 (in seconds)。
    """
    import time
    tic = time.time()
    f(*args)
    toc = time.time()
    return toc - tic

two_loop_time = time_function(classifier.compute_distances_two_loops, X_test)
print('Two loop version took %f seconds' % two_loop_time)

one_loop_time = time_function(classifier.compute_distances_one_loop, X_test)
print('One loop version took %f seconds' % one_loop_time)

no_loop_time = time_function(classifier.compute_distances_no_loops, X_test)
print('No loop version took %f seconds' % no_loop_time)

# You should see significantly faster performance with the fully vectorized implementation!
# [中文] 您应该看到在完全矢量化的操作中, 明显更快的性能 ! @ info: whatsthis

# NOTE: depending on what machine you're using,
# you might not see a speedup when you go from two loops to one loop,
# and might even see a slow-down.
# [中文] 注意: 取决于您使用的机器, 当您从两个循环转到一个循环时, 您可能看不到加速, 甚至可能看到一个缓慢的循环。


### Cross-validation

We have implemented the k-Nearest Neighbor classifier but we set the value k = 5 arbitrarily. We will now determine the best value of this hyperparameter with cross-validation.

---

**中文对照（参考译文，不含解答）**

### 交叉验证

我们已经实现了 k 近邻分类器，但此前只是任意选择了 `k = 5`。现在通过交叉验证确定这个超参数的最佳取值。

In [ ]:
num_folds = 5
k_choices = [1, 3, 5, 8, 10, 12, 15, 20, 50, 100]

X_train_folds = []
y_train_folds = []
################################################################################
# TODO:                                                                        #
# Split up the training data into folds. After splitting, X_train_folds and    #
# y_train_folds should each be lists of length num_folds, where                #
# y_train_folds[i] is the label vector for the points in X_train_folds[i].     #
# Hint: Look up the numpy array_split function.                                #
################################################################################
# [中文] 将训练数据分割成折叠。 分割后, 将数据分割为折叠。 @ info:
# whatsthisX_train_folds和y_train_folds每一页均应是长度列表。 NAME OF
# TRANSLATORSnum_folds,其中y_train_folds[i] 是指用于计算数值的标签矢量。X_train_folds[i] 提示:
# 向上看, 向下看(numpy)array_split函数。


# A dictionary holding the accuracies for different values of k that we find
# when running cross-validation. After running cross-validation,
# k_to_accuracies[k] should be a list of length num_folds giving the different
# accuracy values that we found when using that value of k.
# [中文] 字典中包含我们运行交叉校验时发现的 k 不同值的准确率。 在运行交叉校准后, k_to_accuracies [k] 应该是一个长度列表 num_folds
#, 给出使用 k 的准确率值时发现的不同值。
k_to_accuracies = {}


################################################################################
# TODO:                                                                        #
# Perform k-fold cross validation to find the best value of k. For each        #
# possible value of k, run the k-nearest-neighbor algorithm num_folds times,   #
# where in each case you use all but one of the folds as training data and the #
# last fold as a validation set. Store the accuracies for all fold and all     #
# values of k in the k_to_accuracies dictionary.                               #
################################################################################
# [中文] TODO: 执行 k- 倍交叉验证以找到 k 的最大值。 对于 k 的每一种可能的值, 运行 k- 近邻算法 num_folds 次数, 在每种情况下,
# 您使用全部折叠作为训练数据, 最后折叠作为验证集。 在 k_to_accuracies 字典中保存所有折叠和 k 的所有值。


# Print out the computed accuracies
# [中文] 打印计算精度
for k in sorted(k_to_accuracies):
    for accuracy in k_to_accuracies[k]:
        print('k = %d, accuracy = %f' % (k, accuracy))

In [ ]:
# plot the raw observations
# [中文] 绘制原始观察
for k in k_choices:
    accuracies = k_to_accuracies[k]
    plt.scatter([k] * len(accuracies), accuracies)

# plot the trend line with error bars that correspond to standard deviation
# [中文] 用符合标准差的错误栏绘制趋势行
accuracies_mean = np.array([np.mean(v) for k,v in sorted(k_to_accuracies.items())])
accuracies_std = np.array([np.std(v) for k,v in sorted(k_to_accuracies.items())])
plt.errorbar(k_choices, accuracies_mean, yerr=accuracies_std)
plt.title('Cross-validation on k')
plt.xlabel('k')
plt.ylabel('Cross-validation accuracy')
plt.show()

In [ ]:
# Based on the cross-validation results above, choose the best value for k,
# retrain the classifier using all the training data, and test it on the test
# data. You should be able to get above 28% accuracy on the test data.
# [中文] 基于以上交叉校验结果,选择 k 的最佳值, 使用所有训练数据对分类器进行再训练, 并在测试数据上测试。 您应该能够在测试数据上取得 28% 的准确度以上
# 。
best_k = 1

classifier = KNearestNeighbor()
classifier.train(X_train, y_train)
y_test_pred = classifier.predict(X_test, k=best_k)

# Compute and display the accuracy
# [中文] 计算并显示准确性
num_correct = np.sum(y_test_pred == y_test)
accuracy = float(num_correct) / num_test
print('Got %d / %d correct => accuracy: %f' % (num_correct, num_test, accuracy))

**Inline Question 3**

Which of the following statements about $k$-Nearest Neighbor ($k$-NN) are true in a classification setting, and for all $k$? Select all that apply.
1. The decision boundary of the k-NN classifier is linear.
2. The training error of a 1-NN will always be lower than or equal to that of 5-NN.
3. The test error of a 1-NN will always be lower than that of a 5-NN.
4. The time needed to classify a test example with the k-NN classifier grows with the size of the training set.
5. None of the above.

$\color{blue}{\textit Your Answer:}$


$\color{blue}{\textit Your Explanation:}$



---

**中文对照（参考译文，不含解答）**

**随堂问题 3**

以下关于 $k$ 近邻（$k$-NN）的说法中，哪些在分类任务中对所有 $k$ 都成立？请选择所有适用选项。

1. k-NN 分类器的决策边界是线性的。
2. 1-NN 的训练误差总是小于或等于 5-NN 的训练误差。
3. 1-NN 的测试误差总是小于 5-NN 的测试误差。
4. 使用 k-NN 分类器预测一个测试样本所需的时间会随着训练集规模增大而增长。
5. 以上都不成立。

**你的回答：** [请自行填写]

**你的解释：** [请自行填写]